## Functions to perform semantic preserving transformations of python code

In [1]:
def default_params(): 
    return {
        'cache_dir': '../datax/hugging_face_cache',
        'bert_model': 'microsoft/codebert-base'
    }
params = default_params()


### Variable renaming

In [2]:
import libcst as cst
from redbaron import RedBaron
import copy
import ast
import re
from libcst.metadata import ScopeProvider, MetadataWrapper, ParentNodeProvider
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline
from libcst import (
    SimpleStatementLine,
    Assign,
    AssignTarget,
    BinaryOperation,
    Name,
    FlattenSentinel,
    TrailingWhitespace,
    Newline,
    SimpleWhitespace,
)

2025-02-05 18:48:26.487189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738781306.505945 2971761 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738781306.511716 2971761 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-05 18:48:26.530542: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
###############################################################################
# Transformer for rename_variable_1: First-letter renaming
###############################################################################

class FirstLetterVariableRenamer(cst.CSTTransformer):
    """
    A simple transformer that replaces variable names with their first letter.
    WARNING: This may lead to collisions if two identifiers share the same first letter.
    """
    METADATA_DEPENDENCIES = (ScopeProvider,)

    def leave_Name(
        self, original_node: cst.Name, updated_node: cst.Name
    ) -> cst.BaseExpression:
        # Only replace names longer than one character
        if len(updated_node.value) > 1:
            new_val = updated_node.value[0]
            return updated_node.with_changes(value=new_val)
        return updated_node

def rename_variable_1(code: str) -> str:
    """
    Replace variable names (and other identifiers) with their first letter.
    
    Args:
        code: A string containing Python source code.
    
    Returns:
        The transformed code as a string.
    """
    module = cst.parse_module(code)
    wrapper = MetadataWrapper(module)
    new_module = wrapper.visit(FirstLetterVariableRenamer())
    return new_module.code

In [4]:
###############################################################################
# Transformer for rename_variable_2: CodeBERT-based renaming for variables only
###############################################################################

class CodeBERTVariableRenamer(cst.CSTTransformer):
    """
    Transformer that uses CodeBERT's fill-mask capability to generate new names,
    but only for variable names. It will skip renaming function definitions, class
    definitions, function calls, and attribute accesses.
    
    For each identifier (that is not skipped) longer than one character, a dummy code context is
    constructed and CodeBERT is used to predict a suitable replacement. If the predicted token is
    not a valid identifier, it falls back to using the first letter.
    
    NOTE: This is an experimental approach and may result in unpredictable names.
    """
    # Request both scope and parent metadata.
    METADATA_DEPENDENCIES = (ScopeProvider, ParentNodeProvider)

    def __init__(self):
        model = AutoModelForMaskedLM.from_pretrained(params['bert_model'], cache_dir=params['cache_dir'])
        tokenizer = AutoTokenizer.from_pretrained(params['bert_model'], cache_dir=params['cache_dir'])
        # Initialize the fill-mask pipeline with the loaded model and tokenizer.
        self.fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)
        # Cache substitutions to avoid repeated model calls for the same variable.
        self.substitutions = {}

    def leave_Name(self, original_node: cst.Name, updated_node: cst.Name) -> cst.BaseExpression:
        # Retrieve the parent node using metadata.
        parent = self.get_metadata(ParentNodeProvider, original_node)
        # Skip renaming if the name is used as:
        # - The name of a function definition.
        # - The name of a class definition.
        # - The function being called.
        # - The attribute in an attribute access.
        if isinstance(parent, cst.FunctionDef) and parent.name == original_node:
            return updated_node
        if isinstance(parent, cst.ClassDef) and parent.name == original_node:
            return updated_node
        if isinstance(parent, cst.Call) and parent.func == original_node:
            return updated_node
        if isinstance(parent, cst.Attribute) and parent.attr == original_node:
            return updated_node

        # Process only if the identifier is longer than one character.
        if len(updated_node.value) > 1:
            original_name = updated_node.value
            if original_name not in self.substitutions:
                # Build a dummy context containing the correct mask token (<mask>).
                dummy_context = f"def dummy({original_name}):\n    <mask> = {original_name}"
                try:
                    results = self.fill_mask(dummy_context)
                except Exception as e:
                    print(f"CodeBERT error for '{original_name}': {e}")
                    self.substitutions[original_name] = original_name[0]
                    return updated_node.with_changes(value=self.substitutions[original_name])
                
                if results:
                    new_name = results[0]["token_str"].strip()
                    # Ensure the new name is a valid Python identifier.
                    if not new_name.isidentifier():
                        new_name = original_name[0]
                else:
                    new_name = original_name[0]
                self.substitutions[original_name] = new_name
            return updated_node.with_changes(value=self.substitutions[original_name])
        return updated_node
    
def rename_variable_2(code: str) -> str:
    """
    Replace variable names (and only variable names) using suggestions from CodeBERT.
    
    Args:
        code: A string containing Python source code.
    
    Returns:
        The transformed code as a string.
    """
    module = cst.parse_module(code)
    wrapper = MetadataWrapper(module)
    new_module = wrapper.visit(CodeBERTVariableRenamer())
    return new_module.code

### Expression-level code transformations

In [5]:
###############################################################################
# Transformer for switch_relation: Relational expression switching
###############################################################################

class RelationalExpressionSwitcher(cst.CSTTransformer):
    """
    Transformer that switches relational expressions.

    For example, transforms:
        a < b  -->  b > a

    Only comparisons with a single relational operator are transformed.
    """
    def leave_Comparison(self, original_node: cst.Comparison, updated_node: cst.Comparison) -> cst.BaseExpression:
        # Only handle comparisons with exactly one comparison target.
        if len(updated_node.comparisons) != 1:
            return updated_node

        comp = updated_node.comparisons[0]

        # Mapping of relational operators to their inverted counterparts.
        mapping = {
            cst.LessThan: cst.GreaterThan,
            cst.GreaterThan: cst.LessThan,
            cst.LessThanEqual: cst.GreaterThanEqual,
            cst.GreaterThanEqual: cst.LessThanEqual,
            # For equality and inequality the operator remains the same when operands are swapped.
            cst.Equal: cst.Equal,
            cst.NotEqual: cst.NotEqual,
            cst.Is: cst.Is,
            cst.IsNot: cst.IsNot,
        }

        op_type = type(comp.operator)
        if op_type in mapping:
            new_operator = mapping[op_type]()
        else:
            new_operator = comp.operator

        # Swap the operands:
        # - New left operand becomes the original comparator.
        # - The new comparison target is built using the original left operand.
        new_left = comp.comparator
        new_comparison = cst.ComparisonTarget(
            operator=new_operator,
            comparator=updated_node.left
        )

        return updated_node.with_changes(left=new_left, comparisons=[new_comparison])

def switch_relation(code: str) -> str:
    """
    Transform relational expressions by switching operands and inverting operators.

    For example, 'a < b' becomes 'b > a'.

    Args:
        code: Python source code as a string.

    Returns:
        The transformed code as a string.
    """
    module = cst.parse_module(code)
    wrapper = MetadataWrapper(module)
    new_module = wrapper.visit(RelationalExpressionSwitcher())
    return new_module.code


In [6]:
###############################################################################
# Transformer for unary_2_add: Converting augmented assignments to explicit assignments
###############################################################################

class AugAssignTransformer(cst.CSTTransformer):
    """
    Transformer that converts augmented assignments (e.g., i += 1) into equivalent
    explicit assignments (e.g., i = i + 1).
    """
    def leave_AugAssign(
        self, original_node: cst.AugAssign, updated_node: cst.AugAssign
    ) -> cst.BaseSmallStatement:
        # Map the augmented assignment operator class names to their corresponding binary operator classes.
        operator_mapping = {
            "AddAssign": cst.Add,
            "SubtractAssign": cst.Subtract,
            "MultiplyAssign": cst.Multiply,
            "DivideAssign": cst.Divide,
            "ModuloAssign": cst.Modulo,
            "PowerAssign": cst.Power,
            "FloorDivideAssign": cst.FloorDivide,
            "BitAndAssign": cst.BitAnd,
            "BitOrAssign": cst.BitOr,
            "BitXorAssign": cst.BitXor,
            "LeftShiftAssign": cst.LeftShift,
            "RightShiftAssign": cst.RightShift,
        }
        # Use the operator's class name (e.g., "AddAssign") as the key.
        op_class_name = updated_node.operator.__class__.__name__
        binary_operator_class = operator_mapping.get(op_class_name, None)
        if binary_operator_class is None:
            # If the operator isn't recognized, leave the node unchanged.
            return updated_node

        # Construct a binary operation: <target> <binary_operator> <value>
        new_binary_op = cst.BinaryOperation(
            left=updated_node.target,
            operator=binary_operator_class(),
            right=updated_node.value,
        )
        # Construct an assignment: <target> = (<target> <binary_operator> <value>)
        new_assign = cst.Assign(
            targets=[cst.AssignTarget(target=updated_node.target)],
            value=new_binary_op,
        )
        return new_assign

def add_2_equal(code: str) -> str:
    """
    Transform augmented assignments (e.g., i += 1) into explicit assignments (e.g., i = i + 1).

    Args:
        code: Python source code as a string.

    Returns:
        The transformed code as a string.
    """
    # Parse the code into a CST.
    module = cst.parse_module(code)
    # Wrap the module to preserve formatting.
    wrapper = MetadataWrapper(module)
    # Visit the tree with our AugAssignTransformer.
    new_module = wrapper.visit(AugAssignTransformer())
    return new_module.code

In [7]:
def infix_dividing(code: str) -> str:
    """
    Transform assignments of the form:
        x = L op1 R
    where R is itself a binary expression and its operator has higher precedence than op1,
    by extracting R into a temporary variable.

    For example, the code:
        x = a + b * c  # comment on assignment
    is transformed into:
        temp = b * c
        x = a + temp  # comment on assignment

    All comments and formatting are preserved.

    Args:
        code: The original Python source code.

    Returns:
        The transformed source code as a string.
    """
    red = RedBaron(code)

    # Precedence mapping: operators as strings mapped to a numeric value.
    precedence_map = {
        "+": 1,
        "-": 1,
        "*": 2,
        "/": 2,
        "%": 2,
        "**": 3,
    }

    # Iterate over all assignment nodes.
    for assign in red.find_all("AssignmentNode"):
        # Check that the right-hand side is a binary operator node.
        if assign.value.type != "binary_operator":
            continue

        outer = assign.value
        # Use dumps() to get the string form and split it into tokens.
        tokens_outer = outer.dumps().split()
        if len(tokens_outer) < 3:
            continue  # not an expression like L op R

        # Assume the outer operator is the second token.
        op1 = tokens_outer[1]

        # Access the right-hand side (R). In RedBaron, for a binary operator node:
        #   - .first gives the left operand,
        #   - .second gives the right operand.
        R = outer.second

        # Only proceed if R is itself a binary operator node.
        if R.type != "binary_operator":
            continue

        tokens_R = R.dumps().split()
        if len(tokens_R) < 3:
            continue
        # Assume the operator in R is the second token.
        op_R = tokens_R[1]

        # Check precedence: extract if precedence(op_R) > precedence(op1)
        if precedence_map.get(op_R, 0) <= precedence_map.get(op1, 0):
            continue

        # Build a temporary assignment string: "temp = " + (dump of R)
        temp_assign_str = "temp = " + R.dumps()
        # Create a new node from the string.
        temp_assignment = RedBaron(temp_assign_str)[0]

        # Insert the temporary assignment before the current assignment.
        # Since assign.parent is a NodeList, we get the index of 'assign'
        # and then insert 'temp_assignment' at that index.
        parent_list = assign.parent
        index = parent_list.index(assign)
        parent_list.insert(index, temp_assignment)

        # Replace the original R (the right-hand side of the outer binary expression)
        # with the temporary variable "temp". Simply assigning "temp" creates a NameNode.
        outer.second = "temp"

    # Return the transformed code as a string.
    return red.dumps()


In [8]:
class SwitchEqualExpTransformer(cst.CSTTransformer):
    """
    Transformer that switches the left and right expressions of a simple equality comparison.
    
    For a comparison of the form:
    
        a == b
    
    where the Comparison node has:
      - left: the left-hand side expression, and
      - comparisons: a list with a single ComparisonTarget whose operator is Equal and
        whose comparator is the right-hand side expression,
        
    this transformer returns a new Comparison node equivalent to:
    
        b == a
        
    The operator remains unchanged. Chained comparisons (with more than one operator)
    are not modified.
    """
    def leave_Comparison(
        self, original_node: cst.Comparison, updated_node: cst.Comparison
    ) -> cst.BaseExpression:
        # Process only simple (non-chained) comparisons.
        if len(updated_node.comparisons) != 1:
            return updated_node

        comp_target = updated_node.comparisons[0]
        # Check that the operator is an equality operator.
        if not isinstance(comp_target.operator, cst.Equal):
            return updated_node

        # Use copy.deepcopy to preserve formatting and any attached comments.
        new_left = copy.deepcopy(comp_target.comparator)
        new_comparator = copy.deepcopy(updated_node.left)

        # Create a new ComparisonTarget node with the same operator.
        new_comp_target = comp_target.with_changes(comparator=new_comparator)
        
        # Build a new Comparison node with swapped sides.
        return updated_node.with_changes(
            left=new_left,
            comparisons=[new_comp_target]
        )

def switch_equal_exp(code: str) -> str:
    """
    Switch the two expressions on both sides of a simple equality (==) comparison.
    
    For example, given the code:
    
        a == b
        
    the transformation produces:
    
        b == a
        
    Only simple (non‑chained) equality comparisons are modified; all comments,
    whitespace, and other code remain unchanged.
    
    Args:
        code: A string containing the original Python source code.
    
    Returns:
        A string containing the transformed Python source code.
    """
    # Parse the code into a CST.
    module = cst.parse_module(code)
    # Wrap the module; metadata is not required for this transformation.
    wrapper = MetadataWrapper(module)
    # Apply our transformer.
    new_module = wrapper.visit(SwitchEqualExpTransformer())
    # Return the modified code.
    return new_module.code


### Examples

In [9]:
complex_code = '''
def compare_values(first_value, second_value, third_value):
    if first_value < second_value:
        print("First is less than second")
    elif second_value >= third_value:
        print("Second is greater than or equal to third")
    elif first_value == third_value:
        print("First equals third")
    else:
        print("No simple relation found")

# Assign multiple variables
x, y, z = 5, 10, 15

# Simple relational expression
if x + y < z:
    print("Sum of x and y is less than z")

# Chained comparison (this one will not be switched by our transformer)
if x < y < z:
    print("Chained comparison: x < y < z")

# A function call should not be renamed:
def my_function(a, b):
    return a + b

result = my_function(x, y)
'''

In [10]:
print("=== Original Complex Code ===")
print(complex_code)

print("=== Renamed using rename_variable_1 (first-letter renaming) ===")
print(rename_variable_1(complex_code))

print("=== Renamed using rename_variable_2 (CodeBERT-based renaming) ===")
print(rename_variable_2(complex_code))

=== Original Complex Code ===

def compare_values(first_value, second_value, third_value):
    if first_value < second_value:
        print("First is less than second")
    elif second_value >= third_value:
        print("Second is greater than or equal to third")
    elif first_value == third_value:
        print("First equals third")
    else:
        print("No simple relation found")

# Assign multiple variables
x, y, z = 5, 10, 15

# Simple relational expression
if x + y < z:
    print("Sum of x and y is less than z")

# Chained comparison (this one will not be switched by our transformer)
if x < y < z:
    print("Chained comparison: x < y < z")

# A function call should not be renamed:
def my_function(a, b):
    return a + b

result = my_function(x, y)

=== Renamed using rename_variable_1 (first-letter renaming) ===

def c(f, s, t):
    if f < s:
        p("First is less than second")
    elif s >= t:
        p("Second is greater than or equal to third")
    elif f == t:
        p

/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of RobertaForMaskedLM were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in 


def compare_values(bills, bills, bills):
    if bills < bills:
        print("First is less than second")
    elif bills >= bills:
        print("Second is greater than or equal to third")
    elif bills == bills:
        print("First equals third")
    else:
        print("No simple relation found")

# Assign multiple variables
x, y, z = 5, 10, 15

# Simple relational expression
if x + y < z:
    print("Sum of x and y is less than z")

# Chained comparison (this one will not be switched by our transformer)
if x < y < z:
    print("Chained comparison: x < y < z")

# A function call should not be renamed:
def my_function(a, b):
    return a + b

bills = my_function(x, y)



In [11]:
sample_code = '''
def compute_statistics(values):
    total = 0
    maximum = -float("inf")
    minimum = float("inf")
    for v in values:
        total += v         # Augmented assignment inside loop
        if v > maximum:
            maximum = v
        if v < minimum:
            minimum = v
    count = len(values)
    average = total / count if count != 0 else 0
    return total, count, average, maximum, minimum

data = [10, 20, 30, 40]
t, c, a, m, n = compute_statistics(data)

# Multiple augmented assignments on variables
t -= 5
c += 2
a *= 1.1
m //= 2
n **= 2

i = 0
while i < 5:
    print("Loop iteration:", i)
    i += 1        # Augmented assignment in a loop
'''
print("=== Original Code ===")
print(sample_code)
print("=== Transformed Code (unary_2_add) ===")
print(add_2_equal(sample_code))

=== Original Code ===

def compute_statistics(values):
    total = 0
    maximum = -float("inf")
    minimum = float("inf")
    for v in values:
        total += v         # Augmented assignment inside loop
        if v > maximum:
            maximum = v
        if v < minimum:
            minimum = v
    count = len(values)
    average = total / count if count != 0 else 0
    return total, count, average, maximum, minimum

data = [10, 20, 30, 40]
t, c, a, m, n = compute_statistics(data)

# Multiple augmented assignments on variables
t -= 5
c += 2
a *= 1.1
m //= 2
n **= 2

i = 0
while i < 5:
    print("Loop iteration:", i)
    i += 1        # Augmented assignment in a loop

=== Transformed Code (unary_2_add) ===

def compute_statistics(values):
    total = 0
    maximum = -float("inf")
    minimum = float("inf")
    for v in values:
        total = total + v         # Augmented assignment inside loop
        if v > maximum:
            maximum = v
        if v < minimum:
            mi

In [12]:
complex_code = '''
# Example 1: Extraction occurs because b * c has higher precedence than +
x = a + b * c

# Example 2: Extraction occurs for subtraction combined with division.
y = d - e / f

# Example 3: No extraction because the right operand is not a binary operation.
z = g + h

# Example 4: More complex scenario; only the right binary operation is extracted.
w = m - n * o

# Other code should remain intact.
print("Hello, world!")
def foo():
    a = 10
    b = 20
    c = a + b * 3
    return c
'''
print("=== Original Code ===")
print(complex_code)
print("=== Transformed Code (infix_dividing) ===")
print(infix_dividing(complex_code))

=== Original Code ===

# Example 1: Extraction occurs because b * c has higher precedence than +
x = a + b * c

# Example 2: Extraction occurs for subtraction combined with division.
y = d - e / f

# Example 3: No extraction because the right operand is not a binary operation.
z = g + h

# Example 4: More complex scenario; only the right binary operation is extracted.
w = m - n * o

# Other code should remain intact.
print("Hello, world!")
def foo():
    a = 10
    b = 20
    c = a + b * 3
    return c

=== Transformed Code (infix_dividing) ===

# Example 1: Extraction occurs because b * c has higher precedence than +
temp = b * c
x = a + temp

# Example 2: Extraction occurs for subtraction combined with division.
temp = e / f
y = d - temp

# Example 3: No extraction because the right operand is not a binary operation.
z = g + h

# Example 4: More complex scenario; only the right binary operation is extracted.
temp = n * o
w = m - temp

# Other code should remain intact.
print("Hello, 

In [13]:
original_code = '''
# This is an example.
a = 10
b = 20

# Simple equality comparison:
result = a == b  # check if a equals b

# Another example with extra spaces:
if   x  ==   y:
    print("Equal!")

# A chained comparison (should not be modified):
if a == b == c:
    print("Chained comparison, not switched.")
    
# A non-equality comparison should remain unchanged:
if d != e:
    print("Not equal")
'''
transformed_code = switch_equal_exp(original_code)
print("=== Original Code ===")
print(original_code)
print("=== Transformed Code (switch_equal_exp) ===")
print(transformed_code)

=== Original Code ===

# This is an example.
a = 10
b = 20

# Simple equality comparison:
result = a == b  # check if a equals b

# Another example with extra spaces:
if   x  ==   y:
    print("Equal!")

# A chained comparison (should not be modified):
if a == b == c:
    print("Chained comparison, not switched.")
    
# A non-equality comparison should remain unchanged:
if d != e:
    print("Not equal")

=== Transformed Code (switch_equal_exp) ===

# This is an example.
a = 10
b = 20

# Simple equality comparison:
result = b == a  # check if a equals b

# Another example with extra spaces:
if   y  ==   x:
    print("Equal!")

# A chained comparison (should not be modified):
if a == b == c:
    print("Chained comparison, not switched.")
    
# A non-equality comparison should remain unchanged:
if d != e:
    print("Not equal")



### Real Data Example

In [ ]:
sample_code = """def should_checkpoint(self):
        
        result = self.last_result or {}
        if result.get(DONE) and self.checkpoint_at_end:
            return True
        return (
            self.checkpoint_freq
            and result.get(TRAINING_ITERATION, 0) % self.checkpoint_freq == 0
        )
"""

In [ ]:
print("=== Original Complex Code ===")
print(sample_code)

print("=== Renamed using rename_variable_1 (first-letter renaming) ===")
print(rename_variable_1(sample_code))

print("=== Renamed using rename_variable_2 (CodeBERT-based renaming) ===")
print(rename_variable_2(sample_code))

print("=== Transformed Code (unary_2_add) ===")
print(add_2_equal(sample_code))

print("=== Transformed Code (infix_dividing) ===")
print(infix_dividing(sample_code))

print("=== Transformed Code (switch_equal_exp) ===")
print(switch_equal_exp(sample_code))